# ALS Matrix Factorization

Goal: Learn latent representations of users and products from historical interactions.

Why: Item-based collaborative filtering relies on local similarity relationships.

Matrix factorization learns global patterns across the entire interaction graph and often produces stronger recommendations.

Output

- User embeddings
- Item embeddings
- Personalized recommendations

In [13]:
import pandas as pd
import numpy as np

from scipy.sparse import csr_matrix

from implicit.als import AlternatingLeastSquares

In [14]:
train_cf = pd.read_parquet(
    "../data/train_cf.parquet"
)

train_cf.shape

(1203320, 6)

In [15]:
user_ids = train_cf["reviewerID"].unique()
item_ids = train_cf["asin"].unique()

user_to_idx = {
    user: idx
    for idx, user in enumerate(user_ids)
}

item_to_idx = {
    item: idx
    for idx, item in enumerate(item_ids)
}

In [16]:
rows = train_cf["reviewerID"].map(
    user_to_idx
)

cols = train_cf["asin"].map(
    item_to_idx
)

data = np.ones(
    len(train_cf)
)

In [17]:
interaction_matrix = csr_matrix(
    (
        data,
        (rows, cols)
    ),
    shape=(
        len(user_ids),
        len(item_ids)
    )
)

interaction_matrix.shape

(190963, 62707)

In [18]:
model = AlternatingLeastSquares(
    factors=64,
    regularization=0.01,
    iterations=20,
    random_state=42
)

model.fit(
    interaction_matrix
)

  0%|          | 0/20 [00:00<?, ?it/s]

In [19]:
user_factors = model.user_factors
item_factors = model.item_factors

print(user_factors.shape)
print(item_factors.shape)

(190963, 64)
(62707, 64)


In [20]:
idx_to_item = {
    v:k
    for k,v in item_to_idx.items()
}

idx_to_user = {
    v:k
    for k,v in user_to_idx.items()
}

In [21]:
catalog = pd.read_parquet("../data/catalog.parquet")
def recommend_user(
    user_id,
    n=10
):

    if user_id not in user_to_idx:
        return []

    user_idx = user_to_idx[user_id]

    ids, scores = model.recommend(
        user_idx,
        interaction_matrix[user_idx],
        N=n
    )

    results = []

    for item_idx in ids:

        asin = idx_to_item[item_idx]

        title = catalog.loc[
            catalog["asin"] == asin,
            "title"
        ]

        if len(title):

            results.append(
                (
                    asin,
                    title.iloc[0]
                )
            )

    return results

In [22]:
print(user_factors.shape)
print(item_factors.shape)

(190963, 64)
(62707, 64)


In [23]:
sample_user = (
    train_cf["reviewerID"]
    .iloc[0]
)

recommend_user(
    sample_user,
    n=10
)

[('B000VX6XL6',
  'Kingston 4 GB microSDHC Class 4 Flash Memory Card SDC4/4GBET'),
 ('B005HMKKH4',
  'WD My Passport 2TB Portable External USB 3.0 Hard Drive Storage Black (WDBY8L0020BBK-NESN)'),
 ('B006GWO5WK', ''),
 ('B0074BW614', 'Kindle Fire HD 7&quot;, Dolby Audio, Dual-Band Wi-Fi, 16 GB'),
 ('B003LSTD38',
  'AmazonBasics Hard Carrying Case for My Passport Essential - Black'),
 ('B000JMJWV2', 'Transcend 4 GB Class 6 SDHC Flash Memory Card TS4GSDHC6'),
 ('B000FBK3QK',
  'CyberPower CP1500AVRLCD Intelligent LCD UPS 1500VA 900W AVR Mini-Tower'),
 ('B003FVVMS0',
  'Mediabridge ULTRA Series Subwoofer Cable (15 Feet) - Dual Shielded with Gold Plated RCA to RCA Connectors - White'),
 ('B0098F5W0Q',
  'Fintie Kindle Fire HD 7&quot; (2012 Old Model) Slim Fit Leather Case with Auto Sleep/Wake (will only fit Amazon Kindle Fire HD 7&quot;, Previous Generation) - Orange'),
 ('B00902SFC4', '')]

In [25]:
test = pd.read_parquet(
    "../data/test.parquet"
)

ground_truth = (
    test.groupby("reviewerID")["asin"]
        .apply(set)
        .to_dict()
)

def precision_at_k(
    recommended,
    relevant,
    k=10
):

    hits = len(
        set(recommended[:k])
        &
        set(relevant)
    )

    return hits / k

In [26]:
def recommend_asins(
    user_id,
    n=10
):

    if user_id not in user_to_idx:
        return []

    user_idx = user_to_idx[user_id]

    ids, scores = model.recommend(
        user_idx,
        interaction_matrix[user_idx],
        N=n
    )

    return [
        idx_to_item[i]
        for i in ids
    ]

   

In [41]:
from tqdm import tqdm
import numpy as np
import pandas as pd

def evaluate_als(
    seed,
    sample_size=1000
):

    np.random.seed(seed)

    users = np.random.choice(
        list(ground_truth.keys()),
        size=sample_size,
        replace=False
    )

    precisions = []

    for user in tqdm(
        users,
        desc=f"Seed {seed}"
    ):

        recs = recommend_asins(
            user,
            n=10
        )

        p = precision_at_k(
            recs,
            ground_truth[user],
            k=10
        )

        precisions.append(p)

    return np.mean(precisions)

In [42]:
seeds = [42, 123, 999]

results = []

for seed in seeds:

    score = evaluate_als(
        seed=seed,
        sample_size=1000
    )

    results.append(score)

    print(
        f"Seed {seed}: "
        f"{score:.6f}"
    )

Seed 42:  18%|█▊        | 178/1000 [00:00<00:00, 870.60it/s]

Seed 42: 100%|██████████| 1000/1000 [00:01<00:00, 590.74it/s]


Seed 42: 0.000900


Seed 123: 100%|██████████| 1000/1000 [00:01<00:00, 823.35it/s]


Seed 123: 0.002100


Seed 999: 100%|██████████| 1000/1000 [00:01<00:00, 727.05it/s]

Seed 999: 0.002400


In [43]:

results_df = pd.DataFrame({
    "Seed": seeds,
    "Precision@10": results
})

results_df

,Seed,Precision@10
0,42,0.0009
1,123,0.0021
2,999,0.0024


In [44]:
import pickle

with open(
    "../data/als_model.pkl",
    "wb"
) as f:
    pickle.dump(model, f)

## Results

| Model | Precision@10 |
|---------|---------|
| Popularity | 0.0010 |
| Item-CF | 0.00123 |
| ALS | 0.00180 |

Observations

- ALS achieved the best performance among collaborative approaches.
- Matrix factorization captured global interaction patterns more effectively than local item-item similarity.
- Performance remains limited by sparse implicit-feedback data and the difficulty of predicting a user's exact next interaction.
- Future improvements will incorporate semantic product information through content-based embeddings and hybrid ranking.